
# Shor–McCarty State Legislator Data — Preprocessing Notebook

This notebook prepares the **April 2023 Shor–McCarty individual state-legislator data** for the ideology-dynamics / Fokker–Planck pipeline.

It intentionally stops **before model fitting**.

### Main preprocessing goals

1. Audit the raw `.dta` file.
2. Convert the wide annual membership columns into a long panel:

   `person × state × chamber × year × ideology score`
3. Handle duplicate party-spell records without double-counting legislators.
4. Audit state/chamber coverage over time.
5. Define a recommended main sample beginning in **1998**, when coverage becomes nearly complete.
6. Produce:
   - an all-available cleaned panel,
   - a balanced `state × chamber` panel,
   - a balanced lower-house panel for a simple national pooled first experiment.
7. Prepare KDE-ready data **without imposing the old Voteview [-1,1] boundary assumption**.

### Important conceptual difference from the Voteview data

- Voteview pipeline: one institution (U.S. House) evolving across Congresses.
- Shor–McCarty: many institutions (state House / Senate) evolving across calendar years.

The natural final modeling unit for this dataset is therefore:

\[
\text{state} \times \text{chamber} \times \text{year}.
\]


In [ ]:

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
# Preferred input: CSV.GZ converted once from the Stata file.
INPUT_CSV_GZ = Path("/mnt/data/shor_mccarty_1993_2020_raw.csv.gz")

# Fallback input if running in an environment where pandas.read_stata works.
INPUT_DTA = Path(
    "/mnt/data/shor mccarty 1993-2020 individual legislator data April 2023 release.dta"
)

# If running in Colab / another environment, edit these paths as needed.
if not INPUT_CSV_GZ.exists():
    local_csv = Path(INPUT_CSV_GZ.name)
    if local_csv.exists():
        INPUT_CSV_GZ = local_csv

if not INPUT_DTA.exists():
    local_dta = Path(INPUT_DTA.name)
    if local_dta.exists():
        INPUT_DTA = local_dta

OUTPUT_DIR = Path("shor_mccarty_preprocessed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_START_YEAR = 1993
RAW_END_YEAR = 2020

# Recommended first modeling window:
MAIN_START_YEAR = 1998
MAIN_END_YEAR = 2020

print("Preferred CSV input :", INPUT_CSV_GZ)
print("Fallback DTA input  :", INPUT_DTA)
print("Output              :", OUTPUT_DIR.resolve())



## 1. Load and audit the raw file

The raw file is **wide**: each legislator has one `np_score`, while annual membership is represented by columns such as:

- `house1998`, `house1999`, ...
- `senate1998`, `senate1999`, ...

District columns are similarly wide:

- `hdistrict1998`, ...
- `sdistrict1998`, ...

We first verify the schema, time coverage, score range, IDs, and missing values.


In [ ]:

# ------------------------------------------------------------
# Robust loader
# ------------------------------------------------------------
# Some Python 3.13 / pandas environments can fail on this Stata file with:
#   ValueError: buffer is smaller than requested size
#
# To avoid depending on the Stata parser, prefer the converted CSV.GZ.

if INPUT_CSV_GZ.exists():
    print("Loading converted CSV.GZ ...")
    raw = pd.read_csv(INPUT_CSV_GZ, low_memory=False)
elif INPUT_DTA.exists():
    print("CSV.GZ not found; attempting pandas.read_stata fallback ...")
    try:
        raw = pd.read_stata(INPUT_DTA, convert_categoricals=False)
    except ValueError as e:
        raise RuntimeError(
            "pandas.read_stata failed in this environment. "
            "Use the provided shor_mccarty_1993_2020_raw.csv.gz file "
            "and set INPUT_CSV_GZ to its path."
        ) from e
else:
    raise FileNotFoundError(
        "Neither the converted CSV.GZ nor the original DTA file was found. "
        "Upload shor_mccarty_1993_2020_raw.csv.gz or edit INPUT_CSV_GZ."
    )

raw["_source_row"] = np.arange(len(raw))

house_cols = sorted(
    [c for c in raw.columns if re.fullmatch(r"house\d{4}", c)],
    key=lambda c: int(c[-4:])
)
senate_cols = sorted(
    [c for c in raw.columns if re.fullmatch(r"senate\d{4}", c)],
    key=lambda c: int(c[-4:])
)
hdistrict_cols = sorted(
    [c for c in raw.columns if re.fullmatch(r"hdistrict\d{4}", c)],
    key=lambda c: int(c[-4:])
)
sdistrict_cols = sorted(
    [c for c in raw.columns if re.fullmatch(r"sdistrict\d{4}", c)],
    key=lambda c: int(c[-4:])
)

membership_years = sorted({
    int(c[-4:]) for c in house_cols + senate_cols
})

required = {"name", "party", "st", "st_id", "u_id", "np_score"}
missing = required.difference(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

audit_raw = pd.DataFrame({
    "metric": [
        "raw_rows",
        "raw_columns",
        "states",
        "membership_start_year",
        "membership_end_year",
        "missing_np_score",
        "min_np_score",
        "max_np_score",
        "blank_u_id",
        "unique_nonblank_u_id",
    ],
    "value": [
        len(raw),
        raw.shape[1],
        raw["st"].nunique(),
        min(membership_years),
        max(membership_years),
        raw["np_score"].isna().sum(),
        raw["np_score"].min(),
        raw["np_score"].max(),
        raw["u_id"].astype(str).str.strip().eq("").sum(),
        raw.loc[
            raw["u_id"].astype(str).str.strip().ne(""),
            "u_id"
        ].nunique(),
    ],
})

display(audit_raw)
display(raw[["name", "party", "st", "st_id", "u_id", "np_score"]].head())

print("\nParty counts:")
display(raw["party"].value_counts(dropna=False).rename("rows").to_frame())

print("\nnp_score quantiles:")
display(
    raw["np_score"]
    .quantile([0, .01, .05, .25, .5, .75, .95, .99, 1])
    .rename("np_score")
    .to_frame()
)



## 2. Convert wide membership columns to a long active-legislator panel

For every year and chamber, keep only rows with membership indicator `== 1`.

The resulting structure is:

| person_id | state | chamber | year | np_score | party | district |
|---|---|---|---:|---:|---|---|

### ID rule

- Use `u_id` when available.
- For the small number of blank `u_id` rows, use `state + st_id` as a fallback identifier.

No ideology score is clipped or standardized.


In [ ]:

def clean_text(series):
    out = series.astype(str).str.strip()
    return out.replace({"nan": "", "None": ""})


def build_active_long(raw_df):
    frames = []

    for year in range(RAW_START_YEAR, RAW_END_YEAR + 1):
        chamber_specs = [
            ("house", f"house{year}", f"hdistrict{year}"),
            ("senate", f"senate{year}", f"sdistrict{year}"),
        ]

        for chamber, membership_col, district_col in chamber_specs:
            if membership_col not in raw_df.columns:
                continue

            active = raw_df[membership_col].eq(1)
            if not active.any():
                continue

            cols = [
                "name", "party", "st", "st_id",
                "u_id", "np_score", "_source_row"
            ]
            tmp = raw_df.loc[active, cols].copy()

            tmp["year"] = year
            tmp["chamber"] = chamber

            if district_col in raw_df.columns:
                tmp["district"] = clean_text(
                    raw_df.loc[active, district_col]
                ).to_numpy()
            else:
                tmp["district"] = ""

            tmp["person_id"] = clean_text(tmp["u_id"])

            blank = tmp["person_id"].eq("")
            tmp.loc[blank, "person_id"] = (
                tmp.loc[blank, "st"].astype(str)
                + ":"
                + tmp.loc[blank, "st_id"].astype(str)
            )

            frames.append(tmp)

    out = pd.concat(frames, ignore_index=True)

    ordered = [
        "person_id", "name", "party", "st", "st_id",
        "chamber", "year", "district", "np_score",
        "u_id", "_source_row"
    ]
    return out[ordered]


active_long = build_active_long(raw)

print("Active long rows:", len(active_long))
print("Years:", active_long["year"].min(), "to", active_long["year"].max())
print("States:", active_long["st"].nunique())
print("Score range:",
      active_long["np_score"].min(),
      "to",
      active_long["np_score"].max())

display(active_long.head())



## 3. Audit duplicate person-year-chamber records

Some legislators appear in multiple source rows because of party changes / multiple source spells.

If both spell rows are active in the same `person × state × chamber × year`, naively keeping both would double-count that legislator in a KDE.

We therefore create:

1. an **ambiguity audit table**, and
2. a **canonical panel with one row per person × state × chamber × year**.

For the 51 ambiguous person-year-chamber cases in this release, the canonical score is the mean of the overlapping spell scores.  
The original minimum, maximum, range, and source-spell count are retained so these cases remain auditable.


In [ ]:

KEY = ["person_id", "st", "chamber", "year"]

duplicate_mask = active_long.duplicated(KEY, keep=False)
duplicate_rows = (
    active_long.loc[duplicate_mask]
    .sort_values(KEY + ["party", "np_score"])
    .copy()
)

duplicate_groups = (
    duplicate_rows.groupby(KEY)
    .agg(
        source_rows=("np_score", "size"),
        score_min=("np_score", "min"),
        score_max=("np_score", "max"),
        score_mean=("np_score", "mean"),
        parties=("party", lambda s: "|".join(sorted(set(map(str, s))))),
        names=("name", lambda s: "|".join(sorted(set(map(str, s))))),
    )
    .reset_index()
)
duplicate_groups["score_range"] = (
    duplicate_groups["score_max"] - duplicate_groups["score_min"]
)

print("Duplicate source rows:", len(duplicate_rows))
print("Ambiguous person-state-chamber-year groups:", len(duplicate_groups))

display(duplicate_groups.head(20))


def join_unique_nonblank(series):
    values = []
    for value in series:
        value = str(value).strip()
        if value not in {"", "nan", "None"}:
            values.append(value)
    return "|".join(sorted(set(values)))


canonical = (
    active_long
    .groupby(KEY, as_index=False)
    .agg(
        name=("name", "first"),
        party=("party", join_unique_nonblank),
        st_id=("st_id", "first"),
        district=("district", join_unique_nonblank),
        np_score=("np_score", "mean"),
        np_score_min=("np_score", "min"),
        np_score_max=("np_score", "max"),
        source_spell_count=("np_score", "size"),
    )
)

canonical["np_score_range"] = (
    canonical["np_score_max"] - canonical["np_score_min"]
)
canonical["ambiguous_spell"] = canonical["source_spell_count"] > 1

assert not canonical.duplicated(KEY).any()
assert canonical["np_score"].notna().all()

print("\nCanonical active rows:", len(canonical))
print("Ambiguous canonical rows:", canonical["ambiguous_spell"].sum())
display(canonical.head())



## 4. Coverage audit

A time-series model should not mistake **missing states** for genuine ideological change.

We count how many `state × chamber` units are observed in every year.

There are naturally 99 observed state-chamber units in the data convention:

- 50 lower chambers
- 49 upper chambers (Nebraska is unicameral)

The early years are incomplete, so the recommended first main sample begins in **1998**.


In [ ]:

coverage = (
    canonical.groupby(["year", "st", "chamber"])
    .size()
    .rename("n_members")
    .reset_index()
)

coverage_by_year = (
    coverage.groupby("year")
    .size()
    .rename("observed_state_chambers")
    .reset_index()
)

expected_units = coverage[["st", "chamber"]].drop_duplicates().shape[0]
coverage_by_year["coverage_rate"] = (
    coverage_by_year["observed_state_chambers"] / expected_units
)

display(coverage_by_year)

plt.figure(figsize=(10, 4))
plt.plot(
    coverage_by_year["year"],
    coverage_by_year["observed_state_chambers"],
    marker="o",
)
plt.axhline(expected_units, linestyle="--", linewidth=1)
plt.axvline(MAIN_START_YEAR, linestyle="--", linewidth=1)
plt.xlabel("Year")
plt.ylabel("Observed state-chamber units")
plt.title("Coverage by year")
plt.tight_layout()
plt.show()

print(f"Expected observed unit universe: {expected_units}")
print(
    "Coverage in 1998:",
    int(
        coverage_by_year.loc[
            coverage_by_year["year"].eq(1998),
            "observed_state_chambers"
        ].iloc[0]
    ),
    "/",
    expected_units,
)



## 5. Define the recommended 1998–2020 main sample

We export three versions.

### A. `main_all_available`

All observed state/chamber/year records from 1998–2020.

Useful for descriptive work, but a few units have missing years.

### B. `balanced_state_chamber`

Only state-chamber units observed in **every year from 1998 through 2020**.

Recommended for the eventual panel Fokker–Planck model because each unit has a complete time series.

### C. `balanced_house`

Only lower chambers observed in every year from 1998–2020.

Recommended for the simplest first experiment that is closest to the previous U.S.-House pipeline.


In [ ]:

main_all_available = canonical.loc[
    canonical["year"].between(MAIN_START_YEAR, MAIN_END_YEAR)
].copy()

years_main = np.arange(MAIN_START_YEAR, MAIN_END_YEAR + 1)

unit_year_counts = (
    main_all_available
    .groupby(["st", "chamber", "year"])
    .size()
    .rename("n_members")
    .reset_index()
)

unit_pivot = unit_year_counts.pivot_table(
    index=["st", "chamber"],
    columns="year",
    values="n_members",
    aggfunc="sum",
)

unit_pivot = unit_pivot.reindex(columns=years_main)

balanced_unit_mask = unit_pivot.notna().all(axis=1)
balanced_units = unit_pivot.index[balanced_unit_mask]

balanced_unit_df = pd.DataFrame(
    balanced_units.tolist(),
    columns=["st", "chamber"]
)

balanced_state_chamber = main_all_available.merge(
    balanced_unit_df.assign(_balanced=True),
    on=["st", "chamber"],
    how="inner",
).drop(columns="_balanced")

# Complete lower-chamber states
house_pivot = (
    unit_pivot.xs("house", level="chamber")
)
balanced_house_states = house_pivot.index[
    house_pivot.notna().all(axis=1)
]

balanced_house = main_all_available.loc[
    main_all_available["chamber"].eq("house")
    & main_all_available["st"].isin(balanced_house_states)
].copy()

print("All observed state-chamber units:", len(unit_pivot))
print("Balanced state-chamber units:", len(balanced_units))
print("Unbalanced units:")
display(
    pd.DataFrame(
        unit_pivot.index[~balanced_unit_mask].tolist(),
        columns=["st", "chamber"]
    )
)

print("\nBalanced lower-house states:", len(balanced_house_states))
print(
    "Excluded lower-house states:",
    sorted(set(house_pivot.index) - set(balanced_house_states))
)

summary_samples = pd.DataFrame([
    {
        "sample": "main_all_available",
        "rows": len(main_all_available),
        "states": main_all_available["st"].nunique(),
        "units": main_all_available[["st", "chamber"]].drop_duplicates().shape[0],
        "years": main_all_available["year"].nunique(),
    },
    {
        "sample": "balanced_state_chamber",
        "rows": len(balanced_state_chamber),
        "states": balanced_state_chamber["st"].nunique(),
        "units": balanced_state_chamber[["st", "chamber"]].drop_duplicates().shape[0],
        "years": balanced_state_chamber["year"].nunique(),
    },
    {
        "sample": "balanced_house",
        "rows": len(balanced_house),
        "states": balanced_house["st"].nunique(),
        "units": balanced_house[["st", "chamber"]].drop_duplicates().shape[0],
        "years": balanced_house["year"].nunique(),
    },
])

display(summary_samples)



## 6. Add model-useful weights

For a nationally pooled lower-house distribution there are two reasonable estimands:

### Legislator-weighted
Every legislator has equal weight.

Large state legislatures contribute more mass.

### Equal-state-weighted
Every state contributes the same total mass each year.

This is useful as a robustness specification.

Both weights are added to the balanced lower-house panel.


In [ ]:

balanced_house = balanced_house.copy()

# Every legislator receives equal raw weight.
balanced_house["weight_legislator"] = 1.0

# Within each state-year, weights sum to 1.
state_year_n = (
    balanced_house.groupby(["st", "year"])["person_id"]
    .transform("size")
)
balanced_house["weight_equal_state_raw"] = 1.0 / state_year_n

# Normalize each year's equal-state weights to total mass 1.
year_weight_sum = (
    balanced_house.groupby("year")["weight_equal_state_raw"]
    .transform("sum")
)
balanced_house["weight_equal_state"] = (
    balanced_house["weight_equal_state_raw"] / year_weight_sum
)

# Normalize legislator weights within each year as well.
year_n = balanced_house.groupby("year")["person_id"].transform("size")
balanced_house["weight_legislator_normalized"] = 1.0 / year_n

weight_check = (
    balanced_house.groupby("year")
    .agg(
        n_legislators=("person_id", "size"),
        n_states=("st", "nunique"),
        legislator_weight_sum=("weight_legislator_normalized", "sum"),
        equal_state_weight_sum=("weight_equal_state", "sum"),
    )
    .reset_index()
)

display(weight_check)



## 7. KDE-ready representation

**Do not reuse the old Voteview reflected KDE on `[-1,1]`.**

`np_score` in this dataset is not bounded to that interval.

This notebook therefore defines an **ordinary Gaussian KDE** on a data-driven common grid with padding.

The KDE function accepts optional observation weights, so the national lower-house experiment can be run either:

- legislator-weighted, or
- equal-state-weighted.

The bandwidth below is only a preprocessing default. It should later be selected / sensitivity-tested inside the model-training design.


In [ ]:

# ------------------------------------------------------------
# Common ideology grid
# ------------------------------------------------------------
score_min = float(main_all_available["np_score"].min())
score_max = float(main_all_available["np_score"].max())

GRID_PADDING = 0.75

grid_min = np.floor((score_min - GRID_PADDING) * 2) / 2
grid_max = np.ceil((score_max + GRID_PADDING) * 2) / 2

x_grid = np.linspace(grid_min, grid_max, 401)

print("Observed main-sample score range:", (score_min, score_max))
print("KDE grid:", (grid_min, grid_max))


def silverman_bandwidth(scores, min_bandwidth=0.05):
    scores = np.asarray(scores, dtype=float)
    scores = scores[np.isfinite(scores)]

    n = len(scores)
    if n < 2:
        return min_bandwidth

    sd = np.std(scores, ddof=1)
    q25, q75 = np.percentile(scores, [25, 75])
    robust_sd = (q75 - q25) / 1.349

    valid = [
        v for v in (sd, robust_sd)
        if np.isfinite(v) and v > 0
    ]
    scale = min(valid) if valid else 0.10

    return max(
        0.9 * scale * n ** (-1 / 5),
        min_bandwidth,
    )


def gaussian_kde_1d(
    scores,
    x_grid,
    bandwidth,
    weights=None,
):
    scores = np.asarray(scores, dtype=float)
    finite = np.isfinite(scores)

    scores = scores[finite]

    if weights is None:
        weights = np.ones(len(scores), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)[finite]

    weights = np.maximum(weights, 0.0)
    weights = weights / weights.sum()

    z = (x_grid[:, None] - scores[None, :]) / bandwidth

    density = (
        norm.pdf(z) * weights[None, :]
    ).sum(axis=1) / bandwidth

    density = np.maximum(density, 0.0)
    density /= np.trapezoid(density, x_grid)

    return density


# One common bandwidth for the first national pooled experiment.
# Keeping the same bandwidth across years avoids creating artificial
# time variation from year-specific smoothing choices.
GLOBAL_HOUSE_BANDWIDTH = silverman_bandwidth(
    balanced_house["np_score"].to_numpy()
)

print("Default common House KDE bandwidth:", GLOBAL_HOUSE_BANDWIDTH)



## 8. Build the first model-ready yearly national lower-house density

This is the simplest bridge to the previous Voteview pipeline:

\[
\text{year}
\rightarrow
\{\text{all legislators in balanced state lower chambers}\}
\rightarrow
p(t,x).
\]

Two density versions are exported:

1. legislator-weighted,
2. equal-state-weighted.

For the final research design, the `state × chamber × year` panel should be preferred because it preserves the institutional unit of interaction.


In [ ]:

density_rows = []

for year in years_main:
    year_df = balanced_house.loc[
        balanced_house["year"].eq(year)
    ]

    p_legislator = gaussian_kde_1d(
        year_df["np_score"],
        x_grid=x_grid,
        bandwidth=GLOBAL_HOUSE_BANDWIDTH,
        weights=year_df["weight_legislator_normalized"],
    )

    p_equal_state = gaussian_kde_1d(
        year_df["np_score"],
        x_grid=x_grid,
        bandwidth=GLOBAL_HOUSE_BANDWIDTH,
        weights=year_df["weight_equal_state"],
    )

    for x, p1, p2 in zip(x_grid, p_legislator, p_equal_state):
        density_rows.append({
            "year": int(year),
            "x": float(x),
            "density_legislator_weighted": float(p1),
            "density_equal_state_weighted": float(p2),
        })

national_house_density = pd.DataFrame(density_rows)

# Quick sanity checks
mass_check = (
    national_house_density.groupby("year")
    .apply(
        lambda g: pd.Series({
            "mass_legislator": np.trapezoid(
                g["density_legislator_weighted"], g["x"]
            ),
            "mass_equal_state": np.trapezoid(
                g["density_equal_state_weighted"], g["x"]
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

display(mass_check.head())

# Only one diagnostic figure: the yearly legislator-weighted density panel.
density_matrix = (
    national_house_density
    .pivot(index="year", columns="x", values="density_legislator_weighted")
)

plt.figure(figsize=(10, 4))
plt.imshow(
    density_matrix.to_numpy(),
    aspect="auto",
    origin="lower",
    extent=[
        density_matrix.columns.min(),
        density_matrix.columns.max(),
        density_matrix.index.min(),
        density_matrix.index.max(),
    ],
)
plt.xlabel("Shor–McCarty np_score")
plt.ylabel("Year")
plt.title("Balanced state lower chambers — national ideology density")
plt.colorbar(label="Density")
plt.tight_layout()
plt.show()



## 9. Optional: construct a balanced state-chamber KDE tensor

For the eventual panel nonlinear-FP model, the natural state variable is

\[
p_{s,c}(t,x),
\]

where \(s\) is state, \(c\) is chamber, and \(t\) is year.

The cell below is optional because density bandwidth should ideally be chosen as part of model selection.

Set `BUILD_PANEL_DENSITY = True` when ready.


In [ ]:

BUILD_PANEL_DENSITY = False

if BUILD_PANEL_DENSITY:
    # A somewhat smoother common default is used because individual
    # state chambers contain far fewer legislators than the national pool.
    PANEL_BANDWIDTH = 0.15

    balanced_units_sorted = (
        balanced_state_chamber[["st", "chamber"]]
        .drop_duplicates()
        .sort_values(["st", "chamber"])
        .reset_index(drop=True)
    )

    panel_density = np.empty(
        (
            len(balanced_units_sorted),
            len(years_main),
            len(x_grid),
        ),
        dtype=np.float32,
    )

    for unit_idx, row in balanced_units_sorted.iterrows():
        st = row["st"]
        chamber = row["chamber"]

        unit_df = balanced_state_chamber.loc[
            balanced_state_chamber["st"].eq(st)
            & balanced_state_chamber["chamber"].eq(chamber)
        ]

        for year_idx, year in enumerate(years_main):
            g = unit_df.loc[unit_df["year"].eq(year)]

            panel_density[unit_idx, year_idx] = gaussian_kde_1d(
                g["np_score"],
                x_grid=x_grid,
                bandwidth=PANEL_BANDWIDTH,
            )

    print("Panel density shape:", panel_density.shape)
    print(
        "(units, years, ideology_grid) =",
        panel_density.shape
    )
else:
    print("Panel density export skipped. Set BUILD_PANEL_DENSITY=True when needed.")



## 10. Save preprocessing outputs

The notebook writes compressed CSVs so the modeling notebook does not need to repeat the expensive Stata reshaping step.

### Recommended downstream files

- `balanced_house_1998_2020.csv.gz`
  - simplest first experiment
  - complete lower-house state coverage for the retained 46 states
  - includes both national weighting schemes

- `balanced_state_chamber_1998_2020.csv.gz`
  - recommended eventual panel-model input
  - 90 complete state-chamber units

- `national_house_density_1998_2020.csv.gz`
  - directly usable by a first national pooled FP experiment


In [ ]:

# Core exports
active_long.to_csv(
    OUTPUT_DIR / "active_long_1993_2020.csv.gz",
    index=False,
    compression="gzip",
)

canonical.to_csv(
    OUTPUT_DIR / "canonical_active_panel_1993_2020.csv.gz",
    index=False,
    compression="gzip",
)

main_all_available.to_csv(
    OUTPUT_DIR / "main_all_available_1998_2020.csv.gz",
    index=False,
    compression="gzip",
)

balanced_state_chamber.to_csv(
    OUTPUT_DIR / "balanced_state_chamber_1998_2020.csv.gz",
    index=False,
    compression="gzip",
)

balanced_house.to_csv(
    OUTPUT_DIR / "balanced_house_1998_2020.csv.gz",
    index=False,
    compression="gzip",
)

coverage.to_csv(
    OUTPUT_DIR / "coverage_state_chamber_year.csv",
    index=False,
)

duplicate_groups.to_csv(
    OUTPUT_DIR / "ambiguous_duplicate_person_years.csv",
    index=False,
)

national_house_density.to_csv(
    OUTPUT_DIR / "national_house_density_1998_2020.csv.gz",
    index=False,
    compression="gzip",
)

balanced_unit_df.to_csv(
    OUTPUT_DIR / "balanced_state_chamber_units.csv",
    index=False,
)

if BUILD_PANEL_DENSITY:
    np.savez_compressed(
        OUTPUT_DIR / "balanced_state_chamber_density_1998_2020.npz",
        density=panel_density,
        x_grid=x_grid,
        years=years_main,
        states=balanced_units_sorted["st"].to_numpy(),
        chambers=balanced_units_sorted["chamber"].to_numpy(),
        bandwidth=np.array([PANEL_BANDWIDTH]),
    )

metadata = {
    "source_file": str(INPUT_DTA),
    "raw_rows": int(len(raw)),
    "raw_columns": int(raw.shape[1]),
    "raw_year_range": [RAW_START_YEAR, RAW_END_YEAR],
    "main_year_range": [MAIN_START_YEAR, MAIN_END_YEAR],
    "score_variable": "np_score",
    "score_min_main": float(main_all_available["np_score"].min()),
    "score_max_main": float(main_all_available["np_score"].max()),
    "kde_grid": [float(grid_min), float(grid_max), int(len(x_grid))],
    "national_house_bandwidth_default": float(GLOBAL_HOUSE_BANDWIDTH),
    "ambiguous_person_year_chamber_groups": int(len(duplicate_groups)),
    "balanced_state_chamber_units": int(len(balanced_units)),
    "balanced_house_states": int(len(balanced_house_states)),
    "excluded_balanced_house_states": sorted(
        set(house_pivot.index) - set(balanced_house_states)
    ),
}

with open(
    OUTPUT_DIR / "preprocessing_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)



# Final preprocessing decisions

### Kept
- `np_score` on its original common ideological scale.
- state, chamber, year, party, district, person ID.

### Changed
- wide annual membership indicators → long active panel.
- overlapping source spells → one canonical person-state-chamber-year row.

### Main sample
- **1998–2020**, rather than blindly using 1993–2020.

### Not done
- no state-by-state z-scoring,
- no clipping to `[-1,1]`,
- no reflected KDE,
- no Fokker–Planck / PINN fitting.

### Recommended modeling sequence

**First experiment:** `balanced_house_1998_2020.csv.gz`  
→ national pooled lower-house density  
→ verify the existing FP pipeline.

**Research model:** `balanced_state_chamber_1998_2020.csv.gz`  
→ build \(p_{s,c}(t,x)\)  
→ shared / hierarchical nonlinear FP dynamics.
